# Simple Baseline — leak-safe ElasticNet

**Stage 1 baseline — PMLDL 2026 project.**

## Attribution
This notebook is adapted from the public Kaggle notebook
<https://www.kaggle.com/code/morodertobias/hull-leak-safe-baseline>
for the *Hull Tactical Market Prediction* competition.
It also follows the Hull starter notebook for its feature list and model.

All modelling logic is the original author's. Changes made for this project are
limited to **data loading and results saving**, and every one of them is marked
with an `ADDED (PMLDL Stage 1)` comment:

1. hard-coded `/kaggle/input/...` paths replaced by the shared `src` loader;
2. training data trimmed to exclude the held-out public block (last 180
   `date_id`s), which the original notebooks trained on;
3. the `kaggle_evaluation` inference server replaced by an offline replay of
   that block, since the forecasting phase is out of scope;
4. scoring routed through `src.metrics` and persisted via `src.results`.

No author names, usernames or team identifiers are reproduced here.


In [1]:
# ===================== ADDED (PMLDL Stage 1) =====================
# Shared project interface. Replaces all hard-coded /kaggle/input paths and
# ad-hoc scoring. Original modelling logic below is untouched.
# ==================================================================
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

from src import *          # see INTERFACE.md
import numpy as np, pandas as pd

set_seed()                 # SEED = 42

# --- universal loader: last 180 date_ids held out (public leaderboard block)
hull = load_dataset()
print(hull)

# --- ADDED: materialise a leak-safe train.csv / test.csv pair so the original
# code can keep using read_csv() without pointing at the public block.
PREPARED = RESULTS_DIR / "prepared"; PREPARED.mkdir(parents=True, exist_ok=True)
hull.train.to_csv(PREPARED / "train.csv", index=False)

_pub = hull.public.copy()
for _c in LOOKAHEAD_COLS:                      # rebuild the API's lagged_* columns
    _pub["lagged_" + _c] = _pub[_c].shift(1).fillna(0.0)
_pub["is_scored"] = True
_pub.drop(columns=list(LOOKAHEAD_COLS)).to_csv(PREPARED / "test.csv", index=False)

PUBLIC_FWD = hull.public["forward_returns"].to_numpy()
PUBLIC_RF  = hull.public["risk_free_rate"].to_numpy()
PUBLIC_Y   = hull.public[TARGET].to_numpy()

def replay_public(predict_fn):
    """ADDED: offline stand-in for kaggle_evaluation - feeds the held-out
    public block to predict() one row at a time, exactly as the API would."""
    import polars as pl
    t = pl.read_csv(PREPARED / "test.csv")
    return np.array([float(predict_fn(t[i:i+1])) for i in range(t.height)])


HullData(train=(7862, 98), public=(180, 98), n_raw_features=94)


# Hull - Leak Safe Baseline

- The training data also contain the public test set. It is the last 180 days, see [data description](https://www.kaggle.com/competitions/hull-tactical-market-prediction/data). Let's remove this part overall to get a meaningful score on the current public leaderboard.
- We are supposed to predict the strategy by day, which is something that depends on the forward_return and risk_free_rate (and some overall effect). This makes me wonder if we need to estimate both of them at the same time, and then deriving a certain stratgey. However, in [this discussion answer](https://www.kaggle.com/competitions/hull-tactical-market-prediction/discussion/608349#3299060), this optimization is done analytically (withouth considering penalty effects). Thus let's take these targets as the true targets to be predicted.
- Otherwise we are modelling as simple as in the [Hull Starter Notebook](https://www.kaggle.com/code/laurentlanteigne/hull-starter-notebook): No time effect, same features and same model.
- Note, the training dataset will be updated throughout the competition.

**All comments welcome!**

## Import & Settings

In [2]:
import os
import pathlib
import numpy as np
import pandas as pd
import polars as pl 
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode, iplot
import plotly as py
init_notebook_mode(connected=True) 
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.preprocessing import StandardScaler
# ADDED: kaggle_evaluation / metric.py are competition-runtime only.
# Scoring is routed through the shared interface instead (see setup cell).
from src.metrics import modified_sharpe

def hull_score(solution, submission, row_id_column_name=""):
    return modified_sharpe(submission["prediction"].to_numpy(),
                           solution["forward_returns"].to_numpy(),
                           solution["risk_free_rate"].to_numpy())

In [3]:
# ADDED: path + seed now come from src.config (SEED pinned to 42 project-wide).
BASE_DIR = DATA_DIR
TEST_SKIP = 180
# same features as in hull starter nb
FEATURES = [
    "S2",
    "E2", "E3",
    "P8", "P9", "P10", "P12", "P13",
    "S1", "S5", 
    "I2",
    "U1",
    "U2",
]
INFO_COLS = ["date_id", "forward_returns", "risk_free_rate"]
# model as in hull starter
CV = 10
L1_RATIO = 0.5
ALPHAS = np.logspace(-4, 2, 100)
MAX_ITER = 1000000

## Load data

In [4]:
# ADDED: loaded via the shared loader instead of a raw read_csv.
data = hull.full.copy()
data["U1"] = data["I2"] - data["I1"]
data["U2"] = data["M11"] / ((data["I2"] + data["I9"] + data["I7"]) / 3)
data = data[FEATURES + INFO_COLS].dropna()
data

,S2,E2,E3,P8,P9,P10,P12,P13,S1,S5,I2,U1,U2,date_id,forward_returns,risk_free_rate
505,-0.285790,2.029588,2.752356,1.734923,0.986772,1.983036,-0.162462,0.592262,0.412392,-0.112613,-1.555331,-2.318559,-0.731815,1511,0.003586,0.000195
506,-0.399753,2.045731,2.770683,1.755699,0.987434,1.991912,-0.578615,0.591931,0.402441,-0.011856,-1.542244,-2.305802,-0.220781,1512,0.004851,0.000194
507,0.059127,2.075762,2.806208,1.761389,0.988095,2.004394,-0.781019,0.591601,0.402388,-0.002560,-1.593017,-2.370795,-0.300937,1513,-0.000507,0.000193
508,0.861838,2.071714,2.798832,1.757052,0.988757,1.996753,-0.576658,0.591270,0.385881,-0.433164,-1.611729,-2.384546,-0.474423,1514,-0.001017,0.000194
509,0.633058,2.057450,2.778705,1.754278,0.989418,1.989469,-0.575774,0.590939,0.359528,-0.124108,-1.640988,-2.410829,-0.295663,1515,0.001272,0.000192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8037,-1.195509,1.379103,1.783608,1.953539,0.048280,2.128138,-0.708674,0.616402,0.749919,-0.474489,0.774224,0.427333,0.202994,9043,0.005458,0.000149
8038,-0.994309,1.426299,1.833289,1.960436,0.048942,2.149090,-0.707988,0.323413,0.940689,-0.619673,0.755745,0.409184,0.138923,9044,-0.004565,0.000147
8039,-0.527179,1.369694,1.772236,1.956624,0.053571,2.120745,-0.707302,0.459325,0.967231,-0.463610,0.772152,0.429890,0.016415,9045,0.001852,0.000147
8040,-0.326700,1.388816,1.791988,1.959859,0.054894,2.120585,-0.200284,0.802249,0.986521,-0.402434,0.832127,0.512021,0.115564,9046,0.003463,0.000146


In [5]:
max_train_date = data["date_id"].max() - TEST_SKIP
print("max train date_id:", max_train_date)

max train date_id: 8867


In [6]:
train = data.loc[data["date_id"] <= max_train_date].copy()
test = data.loc[data["date_id"] > max_train_date].copy()
print("train shape:", train.shape)
print("test shape:", test.shape)

train shape: (7357, 16)
test shape: (180, 16)


## Build target
Set target as best strategy on training dataset.

In [7]:
solution = train.copy()
market_excess_returns = solution['forward_returns'] - solution['risk_free_rate']
market_excess_cumulative = (1 + market_excess_returns).prod()
market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solution)) - 1
c = (1 + market_mean_excess_return) ** (1 / (market_excess_returns > 0).mean()) - 1
submission = pd.DataFrame({'prediction': (c / market_excess_returns).clip(0, 2)})
print("best score train:", hull_score(solution, submission, ''))

best score train: 16.986886705807596


In [8]:
train["target"] = submission

In [9]:
fig = go.Figure(data=[go.Scatter3d(
    x=train['forward_returns'],
    y=train['risk_free_rate'],
    z=train['target'],
    mode='markers',
    marker=dict(size=3)
)])
fig.update_layout(
    scene=dict(
        xaxis_title='forward_returns',
        yaxis_title='risk_free_rate',
        zaxis_title='target'
    )
)
iplot(fig)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
market_excess_returns = np.linspace(-0.01, 0.04, 402)
fig = go.Figure(data=[go.Scatter(
    x=market_excess_returns,
    y=(c / market_excess_returns).clip(0, 2),
    mode='markers',
    marker=dict(size=3)
)])
fig.update_layout(
    xaxis_title="market_excess_returns",
    yaxis_title="target",
)
iplot(fig)

## Model

In [ ]:
X_train = train[FEATURES].values
y_train = train["target"].values

In [ ]:
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
model_cv = ElasticNetCV(l1_ratio=L1_RATIO, cv=CV, alphas=ALPHAS, max_iter=MAX_ITER)
model_cv.fit(X_train_scaled, y_train)
model = ElasticNet(alpha=model_cv.alpha_, l1_ratio=L1_RATIO)
model.fit(X_train_scaled, y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the penalty terms. Defaults to 1.0.See the notes for the exact mathematical meaning of thisparameter. ``alpha = 0`` is equivalent to an ordinary least square,solved by the :class:`LinearRegression` object. For numericalreasons, using ``alpha = 0`` with the ``Lasso`` object is not advised.Given this, you should use the :class:`LinearRegression` object.",np.float64(0....7490026177835)
,"l1_ratio l1_ratio: float, default=0.5The ElasticNet mixing parameter, with ``0 <= l1_ratio <= 1``. For``l1_ratio = 0`` the penalty is an L2 penalty. ``For l1_ratio = 1`` itis an L1 penalty. For ``0 < l1_ratio < 1``, the penalty is acombination of L1 and L2.",0.5
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If ``False``, thedata is assumed to be already centered.",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.Check :ref:`an example on how to use a precomputed Gram Matrix in ElasticNet`for details.",False
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


In [ ]:
model.score(X_train_scaled, y_train)

0.00902594679323776

## Invoke on test set

In [ ]:
X_test = test[FEATURES].values
X_test_scaled = sc.transform(X_test)
y_test_pred = model.predict(X_test_scaled)
y_test_pred = np.clip(y_test_pred, 0.0, 2.0)
pd.Series(y_test_pred).describe()

count    180.000000
mean       0.121016
std        0.014500
min        0.070304
25%        0.117442
50%        0.125489
75%        0.130309
max        0.140562
dtype: float64

In [ ]:
solution = test.copy()
submission = pd.DataFrame({'prediction': y_test_pred}, index=solution.index)
print("score public test:", hull_score(solution, submission, ''))

score public test: 0.267273585620148


# Submit

In [ ]:
def predict(test: pl.DataFrame) -> float:
    data = test.to_pandas()
    data["U1"] = data["I2"] - data["I1"]
    data["U2"] = data["M11"] / ((data["I2"] + data["I9"] + data["I7"]) / 3)    
    X = data[FEATURES].values
    X_scaled = sc.transform(X)
    y = model.predict(X_scaled)
    pred = np.clip(y, 0.0, 2.0)[0]
    print(f"date_id: {data['date_id'][0]} -> prediction: {pred:>.4f}")
    return pred

In [ ]:
# ADDED: inference server removed - public phase only, replayed offline in the
# final cell via replay_public(predict).

In [ ]:
MODEL_NAME = "elasticnet_leak_safe"
MODEL_NOTES = "ElasticNet on analytic optimal-allocation target; public block held out"

# ===================== ADDED (PMLDL Stage 1) =====================
# Universal results saver. Replays the held-out public block through predict()
# and records the run in results/leaderboard.csv.
# ==================================================================
allocations = replay_public(predict)

metrics = evaluate(
    PUBLIC_Y,
    allocations,               # allocation doubles as the ranking signal here
    weights=allocations,
    forward_returns=PUBLIC_FWD,
    risk_free_rate=PUBLIC_RF,
)
print({k: round(v, 4) for k, v in metrics.items() if isinstance(v, float)})

save_result(
    model=MODEL_NAME, stage="baseline", metrics=metrics,
    split="public", notes=MODEL_NOTES,
)
display(compare(split="public"))


date_id: 8868 -> prediction: 0.0988
date_id: 8869 -> prediction: 0.1037
date_id: 8870 -> prediction: 0.1039
date_id: 8871 -> prediction: 0.1072
date_id: 8872 -> prediction: 0.1058
date_id: 8873 -> prediction: 0.1053
date_id: 8874 -> prediction: 0.1050
date_id: 8875 -> prediction: 0.1052
date_id: 8876 -> prediction: 0.0988
date_id: 8877 -> prediction: 0.0973
date_id: 8878 -> prediction: 0.0935
date_id: 8879 -> prediction: 0.0899
date_id: 8880 -> prediction: 0.0804
date_id: 8881 -> prediction: 0.0756
date_id: 8882 -> prediction: 0.0703
date_id: 8883 -> prediction: 0.0795
date_id: 8884 -> prediction: 0.0780
date_id: 8885 -> prediction: 0.0816
date_id: 8886 -> prediction: 0.0874
date_id: 8887 -> prediction: 0.0888
date_id: 8888 -> prediction: 0.0877
date_id: 8889 -> prediction: 0.0874
date_id: 8890 -> prediction: 0.0915
date_id: 8891 -> prediction: 0.0876
date_id: 8892 -> prediction: 0.0923
date_id: 8893 -> prediction: 0.0953
date_id: 8894 -> prediction: 0.1032
date_id: 8895 -> prediction:

,model,stage,split,spearman_ic,rmse,modified_sharpe,sharpe,vol_ratio
0,elasticnet_leak_safe,baseline,public,-0.135918,0.121817,0.267274,1.394474,0.104882
